# Retrieval-Augmented Generation

In notebook 01 the model invented a price for Tretwerk's Urban plan: it cannot know a company it was never trained on, and it answers anyway. **Retrieval-augmented generation** (RAG) fixes this without training the model. Before the model answers, a search finds the passages of a knowledge source relevant to the question, and the prompt asks it to answer **from these passages only**.

In this notebook you build a RAG system for Tretwerk's support handbook twice: first from scratch, step by step, then with LangChain and a vector store, the tools a production pipeline uses.

## Learning Objectives

By the end of this notebook, you should be able to:

- Retrieve the passages closest to a question with embeddings.
- Generate answers that are grounded in the retrieved passages and cite them.
- Build a RAG chain with LangChain and Chroma.
- Score an answer for faithfulness and its passages for recall with a model as judge.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

load_dotenv()  # reads GROQ_API_KEY from the .env file
pd.set_option("display.max_colwidth", 120)

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1000,
    max_retries=5,
    reasoning_effort="low",
)

## 1. The Idea

A RAG system works in two parts:

| Part | When | Steps |
|---|---|---|
| **Indexing** | once, and again when the documents change | split the documents into **chunks**, embed each chunk, store the embeddings |
| **Answering** | for every question | embed the question, **retrieve** the closest chunks, put them into the prompt (**augment**), let the model **generate** the answer |

<img src="assets/rag_pipeline.png" alt="Indexing: 20 handbook pages are split into 59 chunks, each chunk is embedded and stored in a vector store. Answering: the question is embedded, the closest chunks are retrieved from the vector store, put into a prompt with rules and the question, and the language model answers with citations." width="1000">

These are **sentence embeddings**: texts with a similar meaning get vectors with a high cosine similarity. That is what lets the search find a passage about *pausing* for a question about *stopping for the winter*, although the two share no word.

## 2. Split the Handbook into Chunks

The handbook has 20 pages, one Markdown file each, in `data/handbook/`. Why not retrieve whole pages? Four reasons to split them into smaller pieces, called **chunks**:

- **One vector blurs several topics.** An embedding is one vector for the whole text. The theft page covers deadlines, deductibles and locks, so its vector mixes all three, and a question about one matches it less closely than a chunk on that topic alone.
- **Embedding models have an input limit.** `all-MiniLM-L6-v2` reads at most 256 tokens and ignores the rest without a warning. The handbook pages are short enough, but a long document would be embedded by its beginning alone.
- **A small result is a precise result.** A chunk that answers the question contains little else, so the rest of the page does not distract the model.
- **Every retrieved token costs tokens in the prompt.** Three chunks leave room in the context and in the rate limit; three whole pages use much more. Notebook 03 measures the difference.

A simple split for this handbook is one chunk per paragraph, since every paragraph covers one topic.

Each chunk starts with its page title. A paragraph like *"With a police report, the customer pays a deductible of 450 euros"* does not say it is about theft; the title does.

In [ ]:
rows = []
for path in sorted((Path("data") / "handbook").glob("*.md")):
    title, body = path.read_text().split("\n", 1)
    title = title.removeprefix("# ")
    for paragraph in body.split("\n\n"):
        if paragraph.strip():
            rows.append({"page": path.stem, "text": f"{title}: {paragraph.strip()}"})

chunks = pd.DataFrame(rows)
print(f"{chunks['page'].nunique()} pages, {len(chunks)} chunks")
chunks.head()

**Result.** 20 pages become 59 chunks. Tables contain no blank lines, so each stays together in one chunk, as row 3 shows for the price table. Every chunk starts with its page title, so even a short paragraph carries its topic.

## 3. Retrieve the Closest Chunks

[`SentenceTransformer`](https://sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html) turns every chunk into one vector. `all-MiniLM-L6-v2` is small (90 MB), fast, and gives 384 numbers per text.

Searching means embedding the question with the same model and ranking the chunks by cosine similarity. This is **dense retrieval**, because it compares dense vectors rather than words.

> **Important.** The question must be embedded with **the same embedding model** as the chunks. Every model learns its own vector space: the same sentence gets completely different numbers from two models, and even two that both return 384 numbers place texts in unrelated positions. A question embedded with another model is compared as if the vectors meant something, so there is no error message, only passages that do not fit. If you change the embedding model, embed every chunk again.

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_vectors = embedder.encode(chunks["text"].tolist())
print(chunk_vectors.shape)

In [ ]:
def search(question, k=3):
    """Return the k chunks with the highest cosine similarity to the question."""
    similarity = embedder.similarity(embedder.encode(question), chunk_vectors)[0]
    return chunks.assign(similarity=similarity.numpy()).nlargest(k, "similarity")


search("Can I stop my subscription for the winter?")

**Result.** The best chunk is the one about **pausing** (0.68), although the question says *stop* and *winter* and the passage says *pause*. Two passages about cancelling follow (0.61 and 0.52): a similar subject, ending the subscription, so their similarity is only slightly lower.

## 4. The Limits of Dense Retrieval

Dense retrieval finds passages by meaning, which is its strength. Three questions show where it is weak.

In [ ]:
search("What is the capital of France?")

**Result.** The handbook says nothing about France, yet the search returns three chunks. Dense retrieval always returns the `k` closest, even when none is close. Their similarity is low, about 0.21, against 0.5 to 0.7 for real matches above, but which value separates relevant from unrelated depends on the model and the documents, so a fixed threshold is unreliable. The prompt in section 5 has to handle passages that do not answer the question.

In [ ]:
search("What if I am not home when the bike is delivered?")

**Result.** All three chunks come from the right page, but in the wrong order. The chunk with the answer (*the technician waits ten minutes; a second attempt costs 25 euros*) is second, at 0.52, just below a general paragraph about delivery at 0.54. The general paragraph shares more words and topics with the question; the answer sits in a single detail. With `k=1`, the model would not get the answer.

In [ ]:
search("How long does the battery last in winter?")

**Result.** The first chunk shares the word *winter* with the question but is about storing the battery indoors. The chunk with the answer, *range drops by up to 30 percent below 5 degrees Celsius*, is second at 0.39. Here too the answer is among the three chunks, one reason to give the model several passages rather than only the best.

## 5. Generate a Grounded Answer

Now the generation step. The prompt numbers the retrieved passages and gives the model three rules: answer only from them, cite the numbers it used, and say so when they do not contain the answer. An answer based only on its sources is **grounded**.

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a support agent at Tretwerk. Answer the question using only the numbered passages from the support handbook. "
                "After each statement, cite the passage number in square brackets, for example [2]. "
                "If the passages do not contain the answer, say that the handbook does not cover it, and nothing else."
            ),
        ),
        ("human", "Passages:\n{passages}\n\nQuestion: {question}"),
    ]
)
answer_chain = rag_prompt | llm | StrOutputParser()


def answer(question):
    """Retrieve and generate; return the answer and the passages it was given."""
    found = search(question)
    passages = "\n\n".join(
        f"[{i}] {text}" for i, text in enumerate(found["text"], start=1)
    )
    return answer_chain.invoke({"passages": passages, "question": question}), found


reply, found = answer("What if I am not home when the bike is delivered?")
print(reply)
found

In [ ]:
for question in [
    "How much does the Urban plan cost per month?",
    "Do you rent e-scooters?",
]:
    reply, found = answer(question)
    print(f"Q: {question}\nA: {reply}\n   ({len(found)} passages)\n")

**Result.** The question from notebook 01 now gets the true price, 49 euros, with a citation, where the model alone invented 29.99 euros. For the e-scooters the model was given three passages, as always, and none is about scooters: it said the handbook does not cover the question instead of guessing. That rule is necessary because dense retrieval always returns passages. The first answer cites passage [2], the second-ranked one: the model found the answer although the search put a more general passage first.

**Your turn:** ask a question whose answer needs passages from **two different pages**, for example what a customer pays who cancels and loses a key, or whether theft protection covers a broken display. Check the answer against the passages it was given: is every statement supported?

In [ ]:
# TODO: a question that needs two pages of the handbook
question = ...

# your code: call answer() and print the reply and the passages

## 6. The Same Pipeline with LangChain

The pipeline above is about 30 lines of code with every step visible. LangChain offers each as a ready-made component:

| Step | From scratch | LangChain |
|---|---|---|
| Documents | rows of a DataFrame | [`Document`](https://reference.langchain.com/python/langchain-core/documents/base/Document) objects with `page_content` and `metadata` |
| Chunks | one per paragraph | [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter) |
| Embeddings | `SentenceTransformer` | [`HuggingFaceEmbeddings`](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings), the same model inside |
| Storage and search | a DataFrame and `similarity` | the vector store [`Chroma`](https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma) and its retriever |
| Generation | `answer_chain` | one LCEL chain from question to answer |

`RecursiveCharacterTextSplitter` cuts a text into chunks of at most `chunk_size` characters, at the most natural boundary it can find. `add_start_index=True` stores where each chunk starts in its document; the next subsection says what for.

In [ ]:
documents = []
for path in sorted((Path("data") / "handbook").glob("*.md")):
    documents.append(
        Document(page_content=path.read_text(), metadata={"page": path.stem})
    )

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=0, add_start_index=True
)
lc_chunks = splitter.split_documents(documents)
print(f"{len(documents)} documents, {len(lc_chunks)} chunks")
print(lc_chunks[20])

Two details are worth knowing:

- `chunk_size` counts **characters**, not tokens. In English, 400 characters are roughly 100 tokens. For a limit in tokens, `RecursiveCharacterTextSplitter.from_tiktoken_encoder` measures with a tokenizer instead.
- The splitter cuts at the largest boundary that fits and falls back through a list of **separators**: an empty line between paragraphs, then a line break, then a space, and finally between any two characters. That fall-back is what *recursive* means. The default list has no sentence boundary, so a long paragraph is cut between words, possibly mid-sentence. You can pass your own, for example `separators=["\n\n", "\n", ". ", " ", ""]`, to cut at a sentence end before falling back to words. For Markdown or code, `RecursiveCharacterTextSplitter.from_language` gives separators that follow headings or functions.

### Chunk Overlap and Start Index

Two settings need a closer look. The cell below splits the theft page into short chunks of 200 characters, once without overlap and once with an overlap of 60. The number before each chunk is its `start_index`, the position of its first character in the page.

In [ ]:
theft_page = documents[6]  # 07_theft.md

for overlap in [0, 60]:
    print(f"chunk_overlap={overlap}")
    small_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200, chunk_overlap=overlap, add_start_index=True
    )
    for chunk in small_splitter.split_documents([theft_page]):
        print(f"  {chunk.metadata['start_index']:4} | {chunk.page_content}")

**Result.** Without overlap the first paragraph is cut after *the full replacement value of*, and the next chunk starts *the bike: 1,800 euros for the Urban U2...*. Retrieved alone, that chunk lists prices without saying they are what a customer pays when there is no police report. The last chunk, *within five working days.*, means nothing alone either. With an overlap of 60 characters the second chunk starts at position 153 instead of 208 and repeats *Without a police report, Tretwerk charges the full replacement value of*, so the prices keep their context and the last chunk becomes a readable sentence. The chunks starting at 311 and 455 are identical in both runs: they begin new paragraphs, and the splitter overlaps only where it cuts one.

**Chunk overlap** repeats the end of a chunk at the start of the next one, whenever the splitter has to cut a paragraph:

| | |
|---|---|
| **For** | A sentence or fact falling on a chunk boundary is still complete in at least one chunk, so it can be found and understood. |
| **Against** | Repeated text means more chunks to embed and store, and two retrieved chunks can share sentences, costing prompt tokens and leaving less room for other passages. |

A common starting point is 10 to 20 percent of `chunk_size`. Text that splits cleanly at paragraphs, like most of this handbook, gains little from it.

**The start index** tells you where a chunk comes from. With the page name in `metadata` it points to the exact place in the source, so an application can show readers the passage behind an answer, restore the original order of retrieved chunks, or fetch the text around one that is too short alone.

A **vector store** keeps the chunks with their embeddings and searches them. Chroma runs inside your Python process; `from_documents` embeds all chunks and stores them. `as_retriever` turns the store into a component that takes a question and returns the `k` closest chunks.

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store = Chroma.from_documents(lc_chunks, embeddings, collection_name="handbook")
retriever = store.as_retriever(search_kwargs={"k": 3})

for chunk in retriever.invoke("Can I stop my subscription for the winter?"):
    print(
        chunk.metadata["page"],
        chunk.metadata["start_index"],
        "|",
        chunk.page_content[:70].replace("\n", " "),
    )

> **Note.** This store is kept in memory and lost when the kernel stops. With `persist_directory` Chroma saves it to disk, so the chunks are embedded only once.

> **Important.** Chroma stores vectors, not the model that made them, so a store opened from disk must be given **the same embedding model** as in section 3.

The chain connects everything. The dictionary at its start runs two things on the same question: the retriever, whose chunks `format_passages` turns into numbered text, and [`RunnablePassthrough`](https://reference.langchain.com/python/langchain-core/runnables/passthrough/RunnablePassthrough), which passes the question on unchanged. The prompt and model are those of section 5.

In [ ]:
def format_passages(chunks):
    """Number the retrieved chunks, as the prompt expects."""
    return "\n\n".join(
        f"[{i}] {chunk.page_content}" for i, chunk in enumerate(chunks, start=1)
    )


rag_chain = (
    {"passages": retriever | format_passages, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(
    rag_chain.invoke(
        "Can I stop my subscription for the winter, and what does it cost?"
    )
)

**Result.** The answer about the pause is correct and cites its passage, but its second half is wrong: the handbook **does** say what a pause costs, 9 euros per month. With `chunk_size=400` the pausing page was split in two, and the retriever returned its first chunk (start index 0), one about cancelling and one about moving, but not the second pausing chunk with the fee. The grounded prompt worked as intended: the model said the information is missing instead of inventing a price. The retrieval failed, and when a RAG answer is wrong or incomplete the retrieved passages are the first thing to look at.

## 7. Scoring the Answer

Reading one answer found the fault above. A support assistant answers thousands of questions, so the check has to become a number. Two questions are worth keeping apart, because they blame different halves of the pipeline:

- **Faithfulness**: is every claim in the answer supported by the passages it was given? This judges the generation.
- **Context recall**: do the passages hold what a correct answer needs? This judges the retrieval.

A language model can check both. Give it the passages and the text to check, and ask it to mark each claim one at a time. A model used this way is a **judge**, and it is what section 5's exercise asked you to do by hand. Libraries such as [Ragas](https://docs.ragas.io/) ship these metrics ready-made; the judge below is written out instead, so a score is something you can read rather than import.

Recall needs a correct answer, written by a person, to compare the passages against. Faithfulness needs no such reference, which is what makes it the cheaper of the two to run on a system that is already live.

In [ ]:
def score(verdicts, good_label):
    """Fraction of the judge's lines carrying the good label, and the lines themselves."""
    lines = [line.strip() for line in verdicts.strip().split("\n") if "|" in line]
    return sum(line.upper().startswith(good_label) for line in lines) / len(lines), lines


question = "Can I stop my subscription for the winter, and what does it cost?"
# what a correct answer would say, written from the handbook by hand
reference = "A pause lasts up to two months per calendar year, and it costs 9 euros per month for storage."

passages = format_passages(retriever.invoke(question))
answer = rag_chain.invoke(question)

In [ ]:
# one line per claim, so the verdicts can be counted without reading prose
judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You check an answer against the passages it was given. Split the answer into "
            "single factual claims and judge each one on the passages alone. Write one line "
            "per claim: SUPPORTED or UNSUPPORTED, then ' | ', then the claim. Nothing else.",
        ),
        ("human", "Passages:\n{passages}\n\nAnswer:\n{answer}"),
    ]
)

faithfulness, claims = score(
    (judge_prompt | llm | StrOutputParser()).invoke(
        {"passages": passages, "answer": answer}
    ),
    "SUPPORTED",
)
print(f"faithfulness {faithfulness:.2f}", *claims, sep="\n")

In [ ]:
# the same line format, but judged against the reference instead of the answer
recall_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You check whether passages contain what a correct answer needs. Split the "
            "correct answer into single factual claims and look for each one in the "
            "passages. Write one line per claim: FOUND or MISSING, then ' | ', then the "
            "claim. Nothing else.",
        ),
        ("human", "Passages:\n{passages}\n\nCorrect answer:\n{reference}"),
    ]
)

recall, needed = score(
    (recall_prompt | llm | StrOutputParser()).invoke(
        {"passages": passages, "reference": reference}
    ),
    "FOUND",
)
print(f"context recall {recall:.2f}", *needed, sep="\n")

**Result.** Context recall is 0.5 in every run: the retriever never returns the chunk with the 9 euro fee, so half of what a correct answer needs is missing, and the number holds still because retrieval is deterministic. Faithfulness moves between runs, because the answer does. When the model refuses to guess, as the run in section 6 did, every claim it makes is supported and faithfulness is 1. When it fills the gap instead and says the pause is free, that sentence comes back `UNSUPPORTED` and the score falls. Read together, the two numbers place the fault: retrieval lost the fee, and whatever the model then says about the price follows from that.

**The same scores from a library.** [Ragas](https://docs.ragas.io/) packages faithfulness, context recall and context precision, runs them over a whole set of questions at once and collects the results in a table. It is the tool to reach for once a project scores more than a handful of answers. It is not installed here, because the point of this section is that a score is counted claims rather than a number out of a box.

## Summary

- **RAG** answers from documents the model was never trained on: retrieve the relevant passages, add them to the prompt, generate. The documents are split into **chunks** and embedded once; each question is embedded and compared with them.
- **Dense retrieval** ranks chunks by the cosine similarity of their embeddings, finding passages by meaning. The question has to be embedded with **the same model** as the chunks, or the similarities are meaningless. It always returns `k` results, even for an unrelated question, and its ranking within a topic can be off.
- A **grounded** prompt tells the model to answer only from numbered passages, to cite them and to admit when the answer is missing.
- LangChain provides the same steps as components: `Document`, `RecursiveCharacterTextSplitter`, `HuggingFaceEmbeddings`, `Chroma` with `as_retriever`, and an LCEL chain.
- When an answer is wrong, check the retrieved passages first: the model cannot use a passage it was never given.
- A model used as a **judge** turns that check into a number. **Faithfulness** asks whether every claim in the answer is supported by its passages, **context recall** whether the passages hold what a correct answer needs. The first blames the generation, the second the retrieval.

## Sources

- Lewis, P. et al. (2020). [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401): the paper that named RAG.
- Reimers, N. and Gurevych, I. (2019). [Sentence-BERT](https://arxiv.org/abs/1908.10084): the sentence embeddings behind dense retrieval.
- Liu, N. F. et al. (2023). [Evaluating Verifiability in Generative Search Engines](https://arxiv.org/abs/2304.09848): human evaluation of generated answers along fluency, utility, citation recall and citation precision.
- [Ragas](https://docs.ragas.io/) packages faithfulness, context recall and related metrics as a library.
- Tretwerk and its handbook are fictional, written for this repository.